In [5]:
import csv
import os

# File paths for storing data
BOOKS_FILE = 'books.csv'
ISSUED_BOOKS_FILE = 'issued_books.csv'

# Global lists to store book and issued book records
books = []
issued_books = []

def load_data():
    """Loads book and issued book data from CSV files."""
    global books, issued_books
    books = []
    issued_books = []

    # Load books
    if os.path.exists(BOOKS_FILE):
        with open(BOOKS_FILE, mode='r', newline='', encoding='utf-8') as file:
            reader = csv.DictReader(file)
            for row in reader:
                try:
                    row['quantity'] = int(row['quantity'])
                    row['book_id'] = row['book_id'] # Keep as string for consistency if user inputs alpha-numeric IDs
                    books.append(row)
                except ValueError:
                    print(f"Warning: Skipping malformed book record: {row}")
    print(f"Loaded {len(books)} books.")

    # Load issued books
    if os.path.exists(ISSUED_BOOKS_FILE):
        with open(ISSUED_BOOKS_FILE, mode='r', newline='', encoding='utf-8') as file:
            reader = csv.DictReader(file)
            for row in reader:
                # Ensure book_id is stored consistently (as string)
                row['book_id'] = row['book_id']
                issued_books.append(row)
    print(f"Loaded {len(issued_books)} issued book records.")

def save_data():
    """Saves current book and issued book data to CSV files."""
    # Save books
    if books:
        with open(BOOKS_FILE, mode='w', newline='', encoding='utf-8') as file:
            fieldnames = ['book_id', 'book_name', 'author', 'quantity']
            writer = csv.DictWriter(file, fieldnames=fieldnames)
            writer.writeheader()
            writer.writerows(books)
        print("Books data saved successfully.")
    else:
        print("No books to save.")
        # Optionally remove the file if no books are left
        if os.path.exists(BOOKS_FILE):
            os.remove(BOOKS_FILE)

    # Save issued books
    if issued_books:
        with open(ISSUED_BOOKS_FILE, mode='w', newline='', encoding='utf-8') as file:
            fieldnames = ['student_name', 'book_id', 'book_name'] # Add book_name for easier display
            writer = csv.DictWriter(file, fieldnames=fieldnames)
            writer.writeheader()
            writer.writerows(issued_books)
        print("Issued books data saved successfully.")
    else:
        print("No issued books to save.")
        # Optionally remove the file if no issued books are left
        if os.path.exists(ISSUED_BOOKS_FILE):
            os.remove(ISSUED_BOOKS_FILE)

def add_book():
    """Adds a new book or updates the quantity of an existing book."""
    print("\n--- Add New Book ---")
    book_id = input("Enter Book ID: ").strip()
    if not book_id:
        print("Book ID cannot be empty.")
        return

    book_name = input("Enter Book Name: ").strip()
    author = input("Enter Author: ").strip()
    try:
        quantity = int(input("Enter Quantity Available: "))
        if quantity < 0:
            print("Quantity cannot be negative.")
            return
    except ValueError:
        print("Invalid quantity. Please enter a number.")
        return

    # Check if book ID already exists
    for book in books:
        if book['book_id'] == book_id:
            book['quantity'] += quantity
            print(f"Book ID '{book_id}' already exists. Quantity updated to {book['quantity']}.")
            save_data()
            return

    # Add new book
    new_book = {
        'book_id': book_id,
        'book_name': book_name,
        'author': author,
        'quantity': quantity
    }
    books.append(new_book)
    print(f"Book '{book_name}' added successfully with ID '{book_id}'.")
    save_data()

def issue_book():
    """Issues a book to a student."""
    print("\n--- Issue Book ---")
    student_name = input("Enter Student Name: ").strip()
    if not student_name:
        print("Student name cannot be empty.")
        return

    book_id = input("Enter Book ID to issue: ").strip()
    if not book_id:
        print("Book ID cannot be empty.")
        return

    found_book = None
    for book in books:
        if book['book_id'] == book_id:
            found_book = book
            break

    if found_book:
        if found_book['quantity'] > 0:
            found_book['quantity'] -= 1
            issued_record = {
                'student_name': student_name,
                'book_id': book_id,
                'book_name': found_book['book_name'] # Store book name for easier report
            }
            issued_books.append(issued_record)
            print(f"Book '{found_book['book_name']}' issued to '{student_name}'.")
            save_data()
        else:
            print("Book is out of stock.")
    else:
        print("Book with this ID not found.")

def return_book():
    """Returns an issued book."""
    print("\n--- Return Book ---")
    book_id = input("Enter Book ID to return: ").strip()
    if not book_id:
        print("Book ID cannot be empty.")
        return

    student_name = input("Enter Student Name who borrowed the book (leave empty if unknown): ").strip()

    issued_record_found = False
    for i, record in enumerate(issued_books):
        if record['book_id'] == book_id and (not student_name or record['student_name'] == student_name):
            # Remove from issued records
            removed_record = issued_books.pop(i)
            issued_record_found = True

            # Increase quantity in main books list
            for book in books:
                if book['book_id'] == book_id:
                    book['quantity'] += 1
                    print(f"Book '{book['book_name']}' returned successfully. Quantity updated.")
                    save_data()
                    return
            # If book was in issued but not in main books (corrupted data)
            print("Warning: Returned book ID found in issued records but not in main book list. Quantity not updated.")
            save_data()
            return

    if not issued_record_found:
        print("No matching issued record found for this Book ID and Student Name (if provided).")
    else:
        print("No matching issued record found for this Book ID.")

def search_book():
    """Searches for books by name or author."""
    print("\n--- Search Book ---")
    search_term = input("Enter Book Name or Author to search: ").strip().lower()
    if not search_term:
        print("Search term cannot be empty.")
        return

    found_books = []
    for book in books:
        if search_term in book['book_name'].lower() or search_term in book['author'].lower():
            found_books.append(book)

    if found_books:
        print("\nSearch Results:")
        for book in found_books:
            print(f"  ID: {book['book_id']}, Name: {book['book_name']}, Author: {book['author']}, Available: {book['quantity']}")
    else:
        print("No books found matching your search term.")

def display_reports():
    """Displays reports on total, issued, and available books."""
    print("\n--- Library Reports ---")

    total_books_count = sum(book['quantity'] for book in books)
    total_unique_books = len(books)
    total_issued_count = len(issued_books)

    print(f"Total unique book titles: {total_unique_books}")
    print(f"Total copies of all books available: {total_books_count}")
    print(f"Total books currently issued: {total_issued_count}")

    print("\n--- Available Books ---")
    if books:
        for book in books:
            print(f"  ID: {book['book_id']}, Name: {book['book_name']}, Author: {book['author']}, Available: {book['quantity']}")
    else:
        print("No books in the library.")

    print("\n--- Issued Books ---")
    if issued_books:
        for record in issued_books:
            print(f"  Student: {record['student_name']}, Book ID: {record['book_id']}, Book Name: {record['book_name']}")
    else:
        print("No books currently issued.")

def display_menu():
    """Displays the main menu options."""
    print("\n--- Library Management System Menu ---")
    print("1. Add New Book / Update Quantity")
    print("2. Issue Book")
    print("3. Return Book")
    print("4. Search Book")
    print("5. Display Reports")
    print("6. Exit")
    print("--------------------------------------")

def main():
    """Main function to run the Library Management System."""
    load_data() # Load data when the program starts

    while True:
        display_menu()
        choice = input("Enter your choice (1-6): ").strip()

        if choice == '1':
            add_book()
        elif choice == '2':
            issue_book()
        elif choice == '3':
            return_book()
        elif choice == '4':
            search_book()
        elif choice == '5':
            display_reports()
        elif choice == '6':
            print("Exiting Library Management System. Goodbye!")
            break
        else:
            print("Invalid choice. Please enter a number between 1 and 6.")

# Run the main program
if __name__ == '__main__':
    main()


Loaded 0 books.
Loaded 0 issued book records.

--- Library Management System Menu ---
1. Add New Book / Update Quantity
2. Issue Book
3. Return Book
4. Search Book
5. Display Reports
6. Exit
--------------------------------------
Enter your choice (1-6): 5

--- Library Reports ---
Total unique book titles: 0
Total copies of all books available: 0
Total books currently issued: 0

--- Available Books ---
No books in the library.

--- Issued Books ---
No books currently issued.

--- Library Management System Menu ---
1. Add New Book / Update Quantity
2. Issue Book
3. Return Book
4. Search Book
5. Display Reports
6. Exit
--------------------------------------
Enter your choice (1-6): 6
Exiting Library Management System. Goodbye!
